# Unit 6 — Data Preparation (BBBC039)

The **inverse** task to Unit 5. We will train a segmentation U-Net to predict the
binary mask from a fluorescence image — same dataset, opposite direction.

Steps:
1. Download images (TIFF, 16-bit) and masks (PNG, instance-encoded)
2. Binarize masks: any instance ID > 0 → foreground
3. Convert 16-bit fluorescence to 8-bit RGB
4. Resize to **512×512**
5. Split **160 train / 40 test** (same RNG seed as Unit 5)
6. Save as
   ```
   datasets/bbbc039_seg/{train,test}/{images,masks}/<basename>.png
   ```


In [ ]:
#df -h --> Check if you have several GBs of space available
#Otherwise delete some cached files
#rm -rf ~/.cache/torch ~/.cache/pip ~/.cache/huggingface/
#uv cache clean

In [ ]:
import os, urllib.request, zipfile, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("Libraries ready")

## 1. Download

BBBC039 — fluorescence microscopy images of U2OS cell nuclei with instance segmentation masks.

In [ ]:
data_dir = "data_download"
os.makedirs(data_dir, exist_ok=True)

urls = {
    "images.zip": "https://data.broadinstitute.org/bbbc/BBBC039/images.zip",
    "masks.zip":  "https://data.broadinstitute.org/bbbc/BBBC039/masks.zip",
}

# If a Unit_5_pix2pixHD/data_download/ already has the zips, link them in
# to avoid a second download.
sibling = "../Unit_5_pix2pixHD/data_download"
for fname in urls:
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        print(f"{fname} already present.")
        continue
    sib_path = os.path.join(sibling, fname)
    if os.path.exists(sib_path):
        os.symlink(os.path.abspath(sib_path), fpath)
        print(f"{fname} linked from Unit_5_pix2pixHD.")
        continue
    print(f"Downloading {fname} ...")
    urllib.request.urlretrieve(urls[fname], fpath)
    print("  done.")

## 2. Extract & inspect

In [ ]:
for fname in ["images.zip", "masks.zip"]:
    with zipfile.ZipFile(os.path.join(data_dir, fname), 'r') as z:
        z.extractall(data_dir)

img_files = sorted([f for f in os.listdir(os.path.join(data_dir, "images")) if f.endswith(".tif")])
mask_files = sorted([f for f in os.listdir(os.path.join(data_dir, "masks"))
                     if f.endswith(".png") and not f.startswith("._")])
print(f"Images: {len(img_files)}, Masks: {len(mask_files)}")

## 3. Raw sample

Images are 696×520, 16-bit grayscale TIFF.
Masks are RGBA PNG; instance IDs live in the **red channel** (0 = background).

In [ ]:
sample = img_files[0]
base = os.path.splitext(sample)[0]
img = Image.open(os.path.join(data_dir, "images", sample))
mask = Image.open(os.path.join(data_dir, "masks", base + ".png"))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(np.array(img), cmap="gray")
ax[0].set_title(f"Fluorescence\n{img.size}, {img.mode}")
ax[1].imshow(np.array(mask)[:, :, 0], cmap="nipy_spectral")
ax[1].set_title("Mask red channel\n(instance IDs)")
ax[2].imshow((np.array(mask)[:, :, 0] > 0).astype(np.uint8), cmap="gray")
ax[2].set_title("Binarized mask")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4. Process & save

* **Image** → min-max normalize 16→8-bit, RGB, resize 512×512 (bilinear)
* **Mask**  → binarize, single-channel L (0/255), resize 512×512 (nearest)
* Files in `images/` and `masks/` share the **same basename** so they're paired by name.

In [ ]:
out_root = "datasets/bbbc039_seg"
for split in ["train", "test"]:
    for sub in ["images", "masks"]:
        os.makedirs(os.path.join(out_root, split, sub), exist_ok=True)

base_to_img  = {os.path.splitext(f)[0]: f for f in img_files}
base_to_mask = {os.path.splitext(f)[0]: f for f in mask_files}
common = sorted(set(base_to_img) & set(base_to_mask))
print(f"Matched pairs: {len(common)}")

random.seed(42)
random.shuffle(common)
train = common[:160]
test  = common[160:]
print(f"Train: {len(train)}, Test: {len(test)}")

TARGET = (512, 512)

def process_and_save(base, split):
    # Fluorescence: 16-bit → 8-bit RGB
    img = Image.open(os.path.join(data_dir, "images", base_to_img[base]))
    arr = np.array(img).astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr = (arr * 255).astype(np.uint8)
    img_rgb = Image.fromarray(arr).convert("RGB").resize(TARGET, Image.BILINEAR)

    # Mask: instance map → binary L (0/255)
    mask = Image.open(os.path.join(data_dir, "masks", base_to_mask[base]))
    r = np.array(mask.split()[0])
    binary = ((r > 0).astype(np.uint8) * 255)
    mask_L = Image.fromarray(binary, mode="L").resize(TARGET, Image.NEAREST)

    img_rgb.save(os.path.join(out_root, split, "images", base + ".png"))
    mask_L.save(os.path.join(out_root, split, "masks", base + ".png"))

for b in train: process_and_save(b, "train")
for b in test:  process_and_save(b, "test")
print("Saved.")

## 5. Verify output

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(6, 8))
train_imgs = sorted(os.listdir(os.path.join(out_root, "train", "images")))
for row, fname in enumerate(train_imgs[:3]):
    img  = Image.open(os.path.join(out_root, "train", "images", fname))
    mask = Image.open(os.path.join(out_root, "train", "masks", fname))
    axes[row, 0].imshow(img, cmap="gray"); axes[row, 0].set_title("Fluor (input)")
    axes[row, 1].imshow(mask, cmap="gray"); axes[row, 1].set_title("Mask (target)")
    for ax in axes[row]: ax.axis("off")
plt.suptitle("Train samples — 512×512", fontsize=13)
plt.tight_layout(); plt.show()

for split in ["train", "test"]:
    ni = len(os.listdir(os.path.join(out_root, split, "images")))
    nm = len(os.listdir(os.path.join(out_root, split, "masks")))
    print(f"{split}: images={ni}, masks={nm}")

## 6. EDA

Mask occupancy = fraction of pixels labelled foreground. A class-imbalance preview.

In [ ]:
occupancies = []
for fname in train_imgs:
    m = np.array(Image.open(os.path.join(out_root, "train", "masks", fname))) / 255.0
    occupancies.append(m.mean())

fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
ax.hist(occupancies, bins=30, color="C2", edgecolor="k")
ax.axvline(np.mean(occupancies), color="r", ls="--", label=f"mean={np.mean(occupancies):.3f}")
ax.set_title("Mask occupancy (train)"); ax.set_xlabel("fraction of foreground pixels")
ax.set_ylabel("count"); ax.legend()
plt.tight_layout(); plt.show()
print(f"Mean foreground fraction = {np.mean(occupancies):.4f} "
      f"(strong background bias — losses will reflect this)")

## Takeaway

Data is ready at `datasets/bbbc039_seg/`. Next: `01_UNet_Training`.